## Objective functions of the thesis "Determination of Optimal Number of Crews for Pipe Replacement Works Using Optimisation Algorithms"

In [ ]:
# Import libraries
import wntr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [162]:
# Import the water network model
ds_sel = 'DS2' # which damage scenario to analyse
inp_file = "BBM-EPS_"+ds_sel+"mcg.inp"
wn = wntr.network.WaterNetworkModel(inp_file)

In [163]:
# Simulation settings
wn.options.time.duration = 3 * 24 * 3600       # 3 days
wn.options.time.hydraulic_timestep = 30*60     # 30 minutes
wn.options.time.report_timestep = 30*60        # 30 minutes

wn.options.hydraulic.demand_model = "PDA"

wn.options.hydraulic.required_pressure = 20      # m
wn.options.hydraulic.minimum_pressure = 0
wn.options.hydraulic.pressure_exponent = 0.5

In [164]:
# timestep
dt = wn.options.time.hydraulic_timestep

# timestep in minutes
dtm = dt/60

In [165]:
# run the simulation
sim = wntr.sim.EpanetSimulator(wn)
results = sim.run_sim()

In [166]:
# getting the actual demand at each node over time 
demand = results.node["demand"]*1000  # convert to L/s


In [167]:
# Find all demand nodes (base demand > 0)
demand_nodes = []

# Expected demand at each node
expected = {}

for node_name in wn.junction_name_list:

    node = wn.get_node(node_name)

    if node.base_demand > 0:

        demand_nodes.append(node_name)
        expected[node_name] = node.base_demand*1000  #L/s

print(f"Demand nodes found: {len(demand_nodes)}")

Demand nodes found: 4201


In [168]:
# Finding emitter nodes (nodes starting with "E_")
emitter_nodes = [
    j for j in wn.junction_name_list
    if j.startswith("E_")
]

# Demand at emitter nodes (leakage) over time
leakage = demand[emitter_nodes]

### 1. OF: Time without supply for hospital/firefighting (FH)
The time that the hospitals and the firefighting flows are without supply It's calculated by multiplying the simulation time step duration  with the number of time steps in which the supply/demand ratio for the hospitals and firefighting flows was less than 0.5.

In [169]:
# Hospitals (node IDs)
hospital_nodes = ["32941", "43816"]

# Fire stations (node IDs)
fire_nodes = ["54537", "21641"]

# Critical nodes (hospitals and fire stations)
critical_nodes = hospital_nodes + fire_nodes

# water demand in critical nodes (base demand)
demand_fh = {}

for node_name in critical_nodes:

    node = wn.get_node(node_name)

    demand_fh[node_name] = node.base_demand*1000
    
# 50% of required demand
ratio_fh = 0.5      

# Dataframe to store undersupplied information for each critical node over time
undersupplied_fh = pd.DataFrame(index=demand.index)

In [170]:
# Calculating FH (undersupplied critical nodes) objective function (OF)
for node in critical_nodes:
    # Base demand required for critical nodes
    DQfh = demand_fh[node]
    # Water supplied to critical nodes 
    Qfh = demand[node]

    undersupplied_fh[node] = (Qfh / DQfh) <= ratio_fh
    
fh_of = undersupplied_fh.sum().sum()*dtm

print(f"FH = {fh_of:,.2f} minutes")       

FH = 4,260.00 minutes


### 2. OF: Resilience loss (RL)
RL considers the Accumulated loss of functionality during the recovery process of the system. It represents the area between the full functionality line (100%) and the functionality time series

In [171]:
# Total demand at all demand nodes over time
Qi_total = demand[demand_nodes].sum(axis=1)

# Total expected demand (base demand) at all demand nodes
DQi_total = sum(expected.values())

# Formula to calculate functionality
functionality = 100 * Qi_total / DQi_total

In [172]:
# Results of the resilience loss OF in %- minutes
rl_of = ((100 - functionality).sum())*dtm

print(f"Resilience loss = {rl_of:,.2f} %-minutes")

Resilience loss = 27,652.40 %-minutes


### 3. OF: Average time of no user service (Time no serv.)
Measures the average time each demand node across the network stayed without service. The formula of this objective is similar to the FH formula, but it considers the sum of all demand nodes that were undersupplied (supply/demand ratio under 0.5) divided by the total amount of demand nodes (DN)

In [173]:
# 50% of required demand
ratio_tns = 0.5

# Dataframe to store undersupplied information for each node over time
undersupplied = pd.DataFrame({
    node: (demand[node] / expected[node]) <= ratio_tns
    for node in demand_nodes
})

# Total number of demand nodes
DN = len(demand_nodes)

# Calculating time no serv. OF
time_no_serv = (undersupplied.sum().sum() * dtm) / DN

print(f"Time no serv. = {(time_no_serv):,.2f} minutes")

Time no serv. = 942.09 minutes


### 4. OF: Number of users without service for eight consecutive hours (NWSECH)
Number of demand nodes that stayed without service for more than eight consecutive hours. It’s calculated by counting the amount of nodes with at least one continuous 8-hour period during which the node's supply/demand ratio never exceeds 0.5.

In [174]:
# Number of required steps based on an 8-hour period and the hydraulic timestep defined in the water network model.
required_steps = int(8 / (dt / 3600))

# This function checks if there are n_steps or more consecutive True values in a given series. 
def has_consecutive_failures(series, n_steps):

#Returns True if there are n_steps or more consecutive True values.

    count = 0

    for value in series:

        if value:

            count += 1

            if count >= n_steps:
                return True

        else:

            count = 0

    return False

In [175]:
# list to store nodes that have no service for 8 consecutive hours (uses the variable "undersupplied" defined above)
nodes_no_service = []

for node in demand_nodes:

    if has_consecutive_failures(undersupplied[node], required_steps):

        nodes_no_service.append(node)
        
# Results of the NWSECH objective function
NWSECH = len(nodes_no_service)

print(f"Nodes without service = {NWSECH}")

Nodes without service = 12


### 5. OF: Water loss (WL)
counts the volume in litres of water lost during the 7-day period after the earthquake, by multiplying the time step with the sum of the outflows across all damages in the system.

In [176]:
# water loss objective function
water_loss = leakage.sum().sum() * dt  # litres

print(f"Total water loss = {water_loss:,.2f} litres")

Total water loss = 76,426,968.00 litres
